# Track C — 01. Mutations Cleaning

Bridge resolution (`profileid -> model_id`) + collapse to one row per `(ensg_id, model_id)`.
Consolidated from the original two-step notebook (profiling + bridge/collapse) into a single
pipeline. Writes three artefacts to `Track - C/outputs/mutations/`.

In [1]:
import os
import numpy as np
import pandas as pd
from data_utils import REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
PROCESSED    = os.path.join(PROJECT_ROOT, 'outputs', 'processed')
OUT_DIR      = os.path.join(PROJECT_ROOT, 'Track - C', 'outputs', 'mutations')
os.makedirs(OUT_DIR, exist_ok=True)

MUTATIONS_PATH = os.path.join(PROCESSED, 'mutations.csv')
PROFILES_PATH  = os.path.join(PROCESSED, 'depmap_profiles.csv')
GENE_LOOKUP    = os.path.join(REF, 'gene_lookup.parquet')
CELL_LOOKUP    = os.path.join(REF, 'cell_line_lookup.parquet')

MUT_PROFILE_COL = 'profileid'
MUT_GENE_COL    = 'ensemblgeneid'
BRIDGE_PROFILE  = 'profileid'
BRIDGE_MODEL    = 'modelid'
CELL_LK_KEY     = 'model_id'
CELL_LK_NAME    = 'cell_line_name'

COORD_COLS   = ['chrom', 'pos', 'ref', 'alt']
IMPACT_COL   = 'vepimpact'
AM_COL       = 'ampathogenicity'
REVEL_COL    = 'revelscore'
AF_COL       = 'af'
DRIVER_BOOLS = ['hessdriver', 'hotspot', 'oncogenehighimpact',
                'tumorsuppressorhighimpact', 'likelylof']
ONCO_COL     = 'oncogenehighimpact'
TSG_COL      = 'tumorsuppressorhighimpact'
VEP_RANK     = {'modifier': 0, 'low': 1, 'moderate': 2, 'high': 3}

DROP_COLS = [
    'gtexgene', 'didaid', 'didaname', 'pharmgkbid', 'brca1funcscore',
    'gwasdisease', 'gwaspmid', 'civicdescription', 'civicid', 'civicscore',
    'hesssignature', 'dbsnprsid', 'rescuereason', 'vepclinsig', 'intron',
    'molecularconsequence', 'nmd', 'ps', 'transcriptlikelylof',
    'lofgenename', 'lofgeneid', 'lofnumberoftranscriptsingene',
    'lofpercentoftranscriptsaffected',
]

def load(path):
    return pd.read_parquet(path) if path.endswith('.parquet') else pd.read_csv(path, low_memory=False)

print('output dir:', OUT_DIR)


output dir: C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mutations


## 1. Load, sparse-tail drop, bridge resolution

In [2]:
mut    = load(MUTATIONS_PATH)
bridge = load(PROFILES_PATH)
clk    = load(CELL_LOOKUP)
glk    = load(GENE_LOOKUP)
print(f'mutations loaded: {len(mut):,} rows x {mut.shape[1]} cols')

mut = mut.drop(columns=[c for c in DROP_COLS if c in mut.columns])
mut[MUT_GENE_COL] = mut[MUT_GENE_COL].str.upper()

bridge_slim = bridge[[BRIDGE_PROFILE, BRIDGE_MODEL]].drop_duplicates(subset=[BRIDGE_PROFILE])
mut = mut.merge(bridge_slim, how='left', left_on=MUT_PROFILE_COL, right_on=BRIDGE_PROFILE)
mut = mut.rename(columns={BRIDGE_MODEL: 'model_id'})
mut['model_id'] = mut['model_id'].str.upper()

unmatched = mut[mut['model_id'].isna()]
valid     = mut[mut['model_id'].notna()].copy()
print(f'rows with model_id:      {len(valid):,}')
print(f'rows unmatched (logged): {len(unmatched):,}  from {unmatched[MUT_PROFILE_COL].nunique()} profiles')


mutations loaded: 1,066,869 rows x 70 cols
rows with model_id:      913,984
rows unmatched (logged): 152,885  from 499 profiles


## 2. Variant dedup + per-variant signal

In [3]:
for c in COORD_COLS:
    valid[c] = valid[c].astype(str)
valid['variant_key'] = valid[COORD_COLS].agg(':'.join, axis=1)

rows_pre = len(valid)
valid = valid.drop_duplicates(subset=[MUT_GENE_COL, 'model_id', 'variant_key'], keep='first')
print(f'variant rows before dedupe: {rows_pre:,}')
print(f'variant rows after  dedupe: {len(valid):,}  ({rows_pre - len(valid):,} removed)')

valid['vep_rank']         = valid[IMPACT_COL].str.lower().map(VEP_RANK)
am    = pd.to_numeric(valid[AM_COL],    errors='coerce') if AM_COL    in valid.columns else pd.Series(dtype=float, index=valid.index)
revel = pd.to_numeric(valid[REVEL_COL], errors='coerce') if REVEL_COL in valid.columns else pd.Series(dtype=float, index=valid.index)
valid['path_max_variant'] = pd.concat([am, revel], axis=1).max(axis=1)
valid['is_high_mod']      = valid['vep_rank'] >= VEP_RANK['moderate']
for b in DRIVER_BOOLS:
    if b in valid.columns:
        valid[b] = valid[b].astype('boolean')


variant rows before dedupe: 913,984
variant rows after  dedupe: 715,123  (198,861 removed)


## 3. Collapse to (ensg_id, model_id) grain

In [4]:
g = valid.groupby([MUT_GENE_COL, 'model_id'], sort=False)

collapsed = pd.DataFrame({
    'max_vep_rank':          g['vep_rank'].max(),
    'max_pathogenicity':     g['path_max_variant'].max(),
    'variant_count':         g['variant_key'].nunique(),
    'multi_hit_high_impact': g['is_high_mod'].sum() >= 2,
    'oncogene_hit':          g[ONCO_COL].any() if ONCO_COL in valid.columns else False,
    'tsg_hit':               g[TSG_COL].any()  if TSG_COL  in valid.columns else False,
    'max_vaf':               g[AF_COL].max()   if AF_COL   in valid.columns else np.nan,
    'variant_ids':           g['variant_key'].apply(list),
}).reset_index()

driver_present = [b for b in DRIVER_BOOLS if b in valid.columns]
any_driver_s = (valid.groupby([MUT_GENE_COL, 'model_id'], sort=False)[driver_present]
                     .any().any(axis=1).reset_index(name='any_driver'))
collapsed = collapsed.merge(any_driver_s, on=[MUT_GENE_COL, 'model_id'], how='left')

collapsed['variant_burden'] = np.log1p(collapsed['variant_count'])
collapsed = collapsed.rename(columns={MUT_GENE_COL: 'ensg_id'})

clk_slim = clk[[CELL_LK_KEY, CELL_LK_NAME]].drop_duplicates(subset=[CELL_LK_KEY])
collapsed = collapsed.merge(clk_slim, how='left', on='model_id')
print(f'collapsed to {len(collapsed):,} (gene, model_id) pairs')


collapsed to 642,491 (gene, model_id) pairs


## 3b. Restrict to gene universe

Only `ensg_id` (the join key) is validated against `gene_lookup` — same pattern as the
fusions universe filter. `detail` carries the same join key and is filtered identically
right before it's written, so the variant-detail table never contains genes the collapsed
table no longer has.

In [5]:
valid_ensg = set(glk['ensg_id'])
print(f'Before universe filter: {len(collapsed):,} rows, {collapsed["ensg_id"].nunique():,} unique genes')

out_of_universe = collapsed[~collapsed['ensg_id'].isin(valid_ensg)]
print(f'Out-of-universe rows: {len(out_of_universe):,} ({len(out_of_universe)/len(collapsed):.2%})')
print(f'Out-of-universe unique genes: {out_of_universe["ensg_id"].nunique():,}')

out_of_universe.to_csv(os.path.join(OUT_DIR, 'mutations_out_of_universe.csv'), index=False)

collapsed = collapsed[collapsed['ensg_id'].isin(valid_ensg)].copy()
print(f'After universe filter: {len(collapsed):,} rows, {collapsed["ensg_id"].nunique():,} unique genes')


Before universe filter: 642,491 rows, 19,576 unique genes
Out-of-universe rows: 5,923 (0.92%)
Out-of-universe unique genes: 1,454
After universe filter: 636,568 rows, 18,122 unique genes


## 4. Write outputs + resolution log

In [6]:
OUT_COLLAPSED = os.path.join(OUT_DIR, 'mutations_collapsed.parquet')
OUT_DETAIL    = os.path.join(OUT_DIR, 'mutations_variant_detail.parquet')
OUT_LOG       = os.path.join(OUT_DIR, 'mutations_resolution_log.csv')

log_rows = [
    {'type': 'unmatched_profile', 'value': p,
     'note': 'profileid absent from depmap_profiles bridge'}
    for p in sorted(unmatched[MUT_PROFILE_COL].dropna().unique())
]
name_gap = collapsed.loc[collapsed[CELL_LK_NAME].isna(), 'model_id'].dropna().unique()
log_rows += [
    {'type': 'model_id_no_name', 'value': m,
     'note': 'model_id present in bridge but absent from cell_line_lookup'}
    for m in sorted(name_gap)
]
log_df = pd.DataFrame(log_rows)

detail_cols = [MUT_GENE_COL, 'model_id', 'variant_key'] + COORD_COLS + [
    c for c in [IMPACT_COL, AM_COL, REVEL_COL, AF_COL, 'proteinchange',
                MUT_PROFILE_COL] + DRIVER_BOOLS if c in valid.columns]
detail = valid[detail_cols].rename(columns={MUT_GENE_COL: 'ensg_id'})
detail = detail[detail['ensg_id'].isin(valid_ensg)].copy()

collapsed.to_parquet(OUT_COLLAPSED, index=False)
detail.to_parquet(OUT_DETAIL, index=False)
log_df.to_csv(OUT_LOG, index=False)

print(f'rows (gene, model_id pairs): {len(collapsed):,}')
print(f'distinct genes:              {collapsed["ensg_id"].nunique():,}')
print(f'distinct cell lines:         {collapsed["model_id"].nunique():,}')
print(f'any_driver True:             {collapsed["any_driver"].sum():,}')
print()
print('written:')
print(f'  {OUT_COLLAPSED}')
print(f'  {OUT_DETAIL}')
print(f'  {OUT_LOG}')


rows (gene, model_id pairs): 636,568
distinct genes:              18,122
distinct cell lines:         1,744
any_driver True:             114,018

written:
  C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mutations\mutations_collapsed.parquet
  C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mutations\mutations_variant_detail.parquet
  C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mutations\mutations_resolution_log.csv
